# `scuffem` tutorial

This notebook teaches the helper functions in `empylib.scuffem` for writing spectral material files and reading SCUFF-EM output tables.

**Learning goals**

- generate spectral input files for SCUFF-EM in a temporary working directory
- read packaged SCUFF-EM scattering outputs into Python objects
- clean raw tables before downstream analysis

**Notebook design**

- every runnable cell calls the public `empylib` API directly
- parameter meanings are explained in markdown and in short inline comments
- outputs are inspected in the same notebook so you can see what each function returns
- the core path is offline-first; internet-backed examples live in clearly marked optional appendices

In [1]:
from pathlib import Path
import os
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "empylib").exists() and (candidate / "docs").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the EMPI Lib repository root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 3)

import tempfile

import empylib.scuffem as scf

DATA_DIR = ROOT / "docs" / "tio2_D700nm"

## Write spectral input files

**Functions used**

- scf.make_spectral_files

**Problem we are solving**

Before a SCUFF-EM run, you need an omega list and material files in the expected format. `make_spectral_files` creates those files from a wavelength grid and a material dictionary.

**Parameter guide for this example**

- `lam`: wavelength grid in micrometers
- `Material`: dictionary mapping material names to complex refractive-index arrays
- the current working directory determines where the files are written

**Outputs to inspect**

- `OmegaList.dat` and one material `.dat` file per dictionary entry

In [2]:
lam = np.linspace(0.45, 1.10, 51)
material_dict = {
    "demo_material": 2.20 + 0.02j + 0 * lam,
}

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    current = Path.cwd()
    try:
        os.chdir(tmp_path)
        scf.make_spectral_files(
            lam,
            material_dict,  # name -> complex refractive-index spectrum
        )
    finally:
        os.chdir(current)

    omega_lines = (tmp_path / "OmegaList.dat").read_text(encoding="utf-8").splitlines()
    material_lines = (tmp_path / "demo_material.dat").read_text(encoding="utf-8").splitlines()

print("First omega lines:", omega_lines[:3])
print("First material lines:", material_lines[:5])

First omega lines: ['13.962634', '13.570595', '13.199969']
First material lines: ['4.650992e+15 4.83960e+00+8.80000e-02i', '4.185892e+15 4.83960e+00+8.80000e-02i', '4.068362e+15 4.83960e+00+8.80000e-02i', '3.957251e+15 4.83960e+00+8.80000e-02i', '3.852048e+15 4.83960e+00+8.80000e-02i']


**How to read the result**

The helper converts your wavelength-based optical data into the text files expected by SCUFF-EM. Using a temporary directory is a good habit for tutorials and tests because it leaves the repository clean.

**Common pitfalls**

- The files are written to the current working directory, so be explicit about where you are
- The material arrays must match the length of the wavelength grid

**Try this next**

- Add a second material to the dictionary and inspect the new `.dat` file
- Use a real material spectrum from `nklib` instead of a constant complex value

## Read SCUFF-EM scattering summaries

**Functions used**

- scf.read_scatter_PFT
- scf.read_avescatter

**Problem we are solving**

After a SCUFF-EM run, you usually want a structured Python object instead of raw text. These readers parse the packaged sample output and return the tables in a convenient form.

**Parameter guide for this example**

- `FileName`: path to a SCUFF-EM output file
- `read_scatter_PFT`: parse the power / force / torque style scattering summary
- `read_avescatter`: parse the angle-averaged scattering summary

**Outputs to inspect**

- `pft_data`: parsed PFT-style content
- `ave_data`: parsed average-scattering content

In [3]:
sample_file = DATA_DIR / "tio2_D700.AVSCAT.EMTPFT"

pft_data = scf.read_scatter_PFT(str(sample_file))
ave_data = scf.read_avescatter(str(sample_file))

print("PFT object type:", type(pft_data))
print("AVE object type:", type(ave_data))
print("First parsed PFT entry:", pft_data[0] if pft_data else "empty")
print("First parsed AVE entry:", ave_data[0] if ave_data else "empty")

PFT object type: <class 'dict'>
AVE object type: <class 'dict'>


KeyError: 0

**How to read the result**

These readers let you move from SCUFF-EM text output into Python data structures quickly, which makes later plotting or comparison work much easier.

**Common pitfalls**

- Use the reader that matches the file structure you have
- If you get an empty output, inspect the raw file to confirm the format is one the helper supports

**Try this next**

- Convert the parsed outputs into a DataFrame for custom plotting
- Compare several SCUFF-EM runs by reading them into a list and stacking the outputs

## Clean a parsed SCUFF-EM table

**Functions used**

- scf.clean_data

**Problem we are solving**

SCUFF-EM output can contain entries you want to standardize or drop before plotting. `clean_data` provides a light post-processing step.

**Parameter guide for this example**

- `data`: parsed SCUFF-EM output such as the result of `read_avescatter`
- `inplace=False`: return a cleaned copy instead of modifying the original object

**Outputs to inspect**

- `cleaned_data`: sanitized version of the parsed SCUFF-EM output

In [4]:
cleaned_data = scf.clean_data(
    ave_data,
    inplace=False,
)

print("Original first entry:", ave_data[0] if ave_data else "empty")
print("Cleaned first entry:", cleaned_data[0] if cleaned_data else "empty")

KeyError: 0

**How to read the result**

A cleaning step is useful when you want downstream plotting or comparisons to operate on consistent column names and value types instead of raw solver text.

**Common pitfalls**

- If you set `inplace=True`, you are modifying the original parsed object
- Cleaning does not replace understanding the original file format; inspect the raw data when something looks wrong

**Try this next**

- Wrap the cleaned output in a DataFrame for plotting
- Save the cleaned table to disk if you want a stable post-processed dataset